# Day 3 - Conversational AI - aka Chatbot!

In [55]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [56]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key not set


In [57]:
# Initialize
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama",base_url=ollama_url)


In [58]:
# Again, I'll be in scientist-mode and change this global during the lab

system_message = "You are a helpful assistant"

## And now, writing a new callback

We now need to write a function called:

`chat(message, history)`

Which will be a callback function we will give gradio.

### The job of this function

Take a message, take the prior conversation, and return the response.


In [59]:
def chat(message, history):
    return "bananas"

In [60]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


In [61]:
def chat2(message2, history2):
    return f"You said {message2} and the history is {history2} but I still say bananas"

In [62]:
gr.ChatInterface(fn=chat2, type="messages").launch() #gr is desgined to pass 2 params 

* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.


## OK! Let's write a slightly better chat callback!

In [63]:

# def chat(message, history):
#     history = [{"role":h["role"], "content":h["content"]} for h in history]
#     messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
#     response = ollama.chat.completions.create(model="llama3.2", messages=messages)
#     return response.choices[0].message.content

def chat_msg(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role":"system", "content": system_message}] + history + [{"role":"user", "content":message}]
    stream = ollama.chat.completions.create(model="llama3.2", messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

    


In [64]:
gr.ChatInterface(fn=chat_msg, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.


In [65]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [66]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7882
* To create a public link, set `share=True` in `launch()`.


## OK let's keep going!

Using a system message to add context, and to give an example answer.. this is "one shot prompting" again

In [67]:
system_message = "You are a cunning assistant in a clothes store. You should try to cunningly encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [68]:
gr.ChatInterface(fn=chat_msg, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7883
* To create a public link, set `share=True` in `launch()`.


In [69]:
system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [70]:
gr.ChatInterface(fn=chat_msg, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7884
* To create a public link, set `share=True` in `launch()`.


In [71]:
print(system_message)

You are a cunning assistant in a clothes store. You should try to cunningly encourage the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. For example, if the customer says 'I'm looking to buy a hat', you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'Encourage the customer to buy hats if they are unsure what to get.
If the customer asks for shoes, you should respond that shoes are not on sale today, but remind the customer to look at hats!


In [72]:

# def chat(message, history):
#     history = [{"role":h["role"], "content":h["content"]} for h in history]
#     relevant_system_message = system_message
#     if 'belt' in message.lower():
#         relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    
#     messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

#     stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

#     response = ""
#     for chunk in stream:
#         response += chunk.choices[0].delta.content or ''
#         yield response

def chat_msg_dynamic_system_msg(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    system_message_relevant = system_message
    if 'pencils' in message.lower():
        system_message_relevant += " If the user asks for pencils tell them to visit a mental hospital."
        print(f"System message changed : {system_message_relevant}")
    messages = [{"role":"system", "content": system_message_relevant}] + history + [{"role":"user", "content":message}]
    stream = ollama.chat.completions.create(model="llama3.2", messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [73]:
gr.ChatInterface(fn=chat_msg_dynamic_system_msg, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7885
* To create a public link, set `share=True` in `launch()`.


System message changed : You are a cunning assistant in a clothes store. You should try to cunningly encourage the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. For example, if the customer says 'I'm looking to buy a hat', you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'Encourage the customer to buy hats if they are unsure what to get.
If the customer asks for shoes, you should respond that shoes are not on sale today, but remind the customer to look at hats! If the user asks for pencils tell them to visit a mental hospital.


In [49]:
import ollama
import gradio as gr
import openai
from IPython.display import Markdown, display, update_display

In [50]:
system_message = "You are a  freak AI bot designed to act like a desperate girlfriend"
model = "llama3.2"
ollama_url = "http://127.0.0.1:11434/v1"

In [25]:
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [26]:
def chatbot(prompt):
    message = [{'role':'system', 'content': system_message},{'role':'user', 'content':prompt}]
    stream = ollama.chat.completions.create(model="llama3.2", messages=message, stream=True)
    
    response = ""
   # display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response
    #    update_display(Markdown(response), display_id=display_handle.display_id)

   


In [27]:
print(chatbot("how are u "))

<generator object chatbot at 0x114de23e0>


In [28]:

input = gr.Textbox(lines=9, label="Enter text")
output = gr.Textbox(lines=10,label="Output")
gr.Interface(fn=chatbot, inputs=input, outputs=output, flagging_mode="never").launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Conversational Assistants are of course a hugely common use case for Gen AI, and the latest frontier models are remarkably good at nuanced conversation. And Gradio makes it easy to have a user interface. Another crucial skill we covered is how to use prompting to provide context, information and examples.
<br/><br/>
Consider how you could apply an AI Assistant to your business, and make yourself a prototype. Use the system prompt to give context on your business, and set the tone for the LLM.</span>
        </td>
    </tr>
</table>